# 2.1 Clean

**Core series, class 3.** Real health data is messy: the same answer written several ways, people who skipped a question, the same measurement in different units, impossible values typed by mistake, and duplicate rows. **Cleaning** turns that into data you can analyse, with every decision written down. In this class we clean the cohort, the survey answers, and the measurements downloaded in **1.1**.

### By the end of this notebook you'll be able to

1. Check that every column has the right **type** (text, number, date)
2. Tell **real answers** from **non-answers** (*PMI: Skip*, *I prefer not to answer*), and turn non-answers into `NA`
3. Pull the useful part out of messy text with `str_extract()` and `case_when()`
4. Find and remove **duplicate** rows, and decide which of several answers to keep
5. Clean measurements using **units** and **plausible ranges**, after looking at the values unit by unit
6. Explain what **missing data** costs, and why not to drop every incomplete row
7. Keep a **cleaning log**: how many rows each step removed

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Which values can we trust, and what should we do with the rest? |
| 🧪 **The analysis** | Counting labels and values, then rules: recode non-answers, extract answers, remove duplicates, keep plausible measurements |
| 🎯 **Why** | Every later result depends on it. An impossible BMI of 999, or "PMI: Skip" counted as an answer, can change a mean, a model, or a conclusion |
| ✅ **What we conclude** | Clean versions of the three files, and a log of what was removed and why |
| ⚠️ **What it can't tell us** | Whether a **plausible** value is **correct**: cleaning removes the impossible, not every error |
| 🔗 **Connections** | Uses the files from **1.1**; **2.2** reshapes the clean files; **2.3** joins them; **3.1–3.4** analyse them |

| | |
|---|---|
| 📂 **Uses** | `cohort_sample.csv`, `survey_raw.csv`, `measurement_raw.csv` (from **1.1**) |
| 💾 **Saves** | `cohort_clean.csv`, `survey_clean.csv`, `measurement_clean.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Load and check types | 10 |
| 3. The cohort: non-answers | 15 |
| 4. The survey answers | 20 |
| 5. The measurements | 30 |
| 6. Missing data | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 105 minutes |
| ⏱️ **Run time** | about **2 minutes** to run every cell. The measurement file is the largest (about 1.8 million rows) |
| 💰 **Data cost** | **$0**: no BigQuery queries; this class reads the files saved in **1.1** and **1.2** from your bucket |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.50** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Clean data, in one table

| Problem | Example from our data | What we do |
|---|---|---|
| **Wrong type** | an ID read as a number; a date read as text | set types when reading the file |
| **Non-answers** | *PMI: Skip*, *I prefer not to answer*, *No matching concept* | turn into `NA` (missing), and say so |
| **Messy text** | *"Are you still seeing a doctor … (PTSD)? - Yes"* | extract the part we need: *Yes* |
| **Duplicates** | the same measurement row twice; two answers to one question | remove exact copies; choose a rule for the rest (e.g. the latest) |
| **Units** | BMI recorded in "percent"; HbA1c in "gram per deciliter" | look at the values for each unit; keep, convert, or drop |
| **Impossible values** | BMI of 0 or 999; systolic blood pressure of 5 | keep only a **plausible range**, chosen before looking at results |
| **Missing data** | no LDL test for many people | keep the `NA`; deal with it per analysis |

### `NA` is not "No"

`NA` means **"we don't know"**. In *All of Us* surveys, someone may have no answer because they **skipped** the question, **preferred not to answer**, or **were never asked** (follow-up questions only appear after certain answers: **skip logic**). None of these means "No". Turning them into "No" would be making up data.

### Write every decision down

Cleaning involves judgment calls ("is a BMI of 75 possible?"). Good practice:

- **Decide the rules before looking at the results** of the analysis, so the rules can't be chosen to get the answer you want
- **Count** what each rule removes (a **cleaning log**)
- **Report** the rules, so others can repeat or question them

### Functions in this class

| Function | What it does |
|---|---|
| `count(x)` | how many rows have each value: the first look at any column |
| `case_when(rule ~ value, …)` | recode values by rules; the first rule that matches wins |
| `%in%` | is the value in this list? |
| `str_extract(text, pattern)` | pull out the part of the text that matches a pattern |
| `distinct()` | remove exact duplicate rows |
| `slice_max(x, n = 1)` | keep the row with the largest `x` in each group (e.g. the latest date) |
| `between(x, low, high)` | is `x` from `low` to `high`? |

## Part 2. Load and check types

`load_data()` reads a CSV from your bucket. `col_types = cols(person_id = "c")` reads the IDs as **text**: IDs are labels, not numbers, and as text they match the IDs in every other file. Dates written like `2019-04-12` are recognised as **dates** automatically.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort          <- load_data("cohort_sample.csv",   col_types = cols(person_id = "c"))
survey_raw      <- load_data("survey_raw.csv",      col_types = cols(person_id = "c"))
measurement_raw <- load_data("measurement_raw.csv", col_types = cols(person_id = "c"))

c(cohort = nrow(cohort), survey = nrow(survey_raw), measurements = nrow(measurement_raw))

# Each column's type: chr = text, dbl = number, date, lgl = TRUE/FALSE
sapply(cohort, function(x) class(x)[1])

**🔍 Read the output:** 10,000 people; about 1,000 survey answers; and about 1.8 million measurements. In the type list, `person_id` should be `character`, the `_date` columns `Date`, and `age` a number.

If a date column says `logical`, the column was **empty** in the file (e.g. no one in the sample has died): R can't tell it's a date. `as.Date()` fixes that.

In [ ]:
cohort <- cohort %>%
  mutate(across(c(date_of_birth, death_date, first_ehr_date, last_ehr_date, first_t2d_date), as.Date))

## Part 3. The cohort: non-answers

Demographic columns come from survey answers, so they include **non-answers**. First, look at every label in `sex_at_birth` and `race` (labels and counts, not people).

In [ ]:
cohort %>% count(sex_at_birth, sort = TRUE) %>% hide_small_counts()
cohort %>% count(race, sort = TRUE) %>% hide_small_counts()
cohort %>% count(ethnicity, sort = TRUE) %>% hide_small_counts()

**🔍 Read the output:** besides real answers, you'll see labels like *PMI: Skip*, *I prefer not to answer*, *None Indicated*, or *No matching concept*. These aren't categories of people: they say **why the answer is missing**.

In our test run there were also odd-looking labels such as *Sex At Birth: Sex At Birth None Of These* and *What Race Ethnicity: Race Ethnicity None Of These*. Despite the labels, these are **real answers** ("none of these describe me"), so we keep them.

**Decision:** we treat these as **non-answers** and set them to `NA`. We keep real answers that are less common (e.g. *More than one population*, *Another single population*), because they describe people.

`NON_ANSWERS` is our list. `if_else(x %in% NON_ANSWERS, NA, x)` replaces any value in the list with `NA`. `across()` does it for several columns at once.

In [ ]:
NON_ANSWERS <- c("PMI: Skip", "PMI: Prefer Not To Answer", "I prefer not to answer", "Prefer not to answer",
                 "None Indicated", "None", "No matching concept", "PMI: Dont Know", "I don't know")

cohort_clean <- cohort %>%
  mutate(across(c(sex_at_birth, gender, race, ethnicity, self_reported_category),
                ~ if_else(.x %in% NON_ANSWERS, NA_character_, .x)),
         t2d = if_else(cohort == "case", "Yes", "No"))     # the outcome, as Yes / No

# How many are now missing?
cohort_clean %>%
  summarise(across(c(sex_at_birth, race, ethnicity), ~ sum(is.na(.x))))

**🔍 Read the output:** how many people now have a missing value for each column. These people stay in the data; they just won't count in analyses of that column.

We also added `t2d` (*Yes* for cases, *No* for controls): the **flag** that every analysis compares.

### Ages

1.1 already kept ages 18 to 100. A quick check:

In [ ]:
summary(cohort_clean$age)

save_data(cohort_clean, "cohort_clean.csv")

## Part 4. The survey answers

Recall from 1.1: each answer **repeats the question**, then " - ", then the answer, and non-answers like *PMI: Skip* stand alone:

In [ ]:
survey_raw %>% count(question_concept_id, answer) %>% hide_small_counts()

### Short question names

Long questions are hard to use as column names. We give each question a **short name** by its concept ID, which never changes, rather than by its text, which could be reworded:

| Concept ID | Short name | Question |
|---|---|---|
| 43528846 | `ptsd_treatment` | currently prescribed medications and/or receiving treatment for PTSD? |
| 43530361 | `ptsd_doctor` | still seeing a doctor or health care provider for PTSD? |

### Extract the answer

`str_extract(answer, "(?<= - ).*$")` takes everything **after** " - " to the end. The pattern is a **regular expression**: `(?<= - )` means "just after ' - '", `.*` "any characters", `$` "the end". For *PMI: Skip* there's no " - ", so the result is `NA`: exactly what we want for a non-answer.

In [ ]:
survey_clean <- survey_raw %>%
  mutate(question = case_when(question_concept_id == 43528846 ~ "ptsd_treatment",
                              question_concept_id == 43530361 ~ "ptsd_doctor"),
         answer_clean = str_extract(answer, "(?<= - ).*$"))

# Check the recoding: every original label next to its clean value
survey_clean %>% count(question, answer_clean) %>% hide_small_counts()

**🔍 Read the output:** each question now has *Yes*, *No*, and `NA` (the non-answers). **Always check a recoding like this**, by counting the new values: a typo in a pattern can silently turn everything into `NA`.

### Duplicates: more than one answer

Some people answered the same question **more than once** (e.g. they took the survey again later). Before deciding what to do, count them:

In [ ]:
survey_clean %>%
  count(person_id, question) %>%
  count(answers_per_person = n, name = "people") %>%
  rename(n = people) %>%
  hide_small_counts()

**🔍 Read the output:** most people have **1** answer per question. If any have 2 or more, we need a rule.

**Decision:** keep each person's **latest** real answer (the most recent `survey_datetime`), after dropping non-answers. `slice_max(survey_datetime, n = 1, with_ties = FALSE)` keeps the latest row in each `person_id` × `question` group.

In [ ]:
survey_final <- survey_clean %>%
  filter(!is.na(answer_clean)) %>%
  group_by(person_id, question) %>%
  slice_max(survey_datetime, n = 1, with_ties = FALSE) %>%
  ungroup() %>%
  select(person_id, question, answer = answer_clean, survey_datetime)

c(before = nrow(survey_raw), after = nrow(survey_final))
survey_final %>% count(question, answer) %>% hide_small_counts()

save_data(survey_final, "survey_clean.csv")

**🔍 Read the output:** fewer rows than we started with (non-answers and repeat answers removed), and at most one *Yes* or *No* per person per question.

### ✍️ Your turn

What share of our **10,000** people answered the doctor question (`ptsd_doctor`) at all? What does that tell you about using it in a model?

<details>
<summary><b>Show answer</b></summary>

```r
n_distinct(survey_final$person_id[survey_final$question == "ptsd_doctor"]) / nrow(cohort_clean)
```

About **5%** in our test run. The question was only asked of people who reported PTSD (**skip logic**), so for 95% the answer is unknown, not "No". A model that requires this variable would drop almost everyone; a model that treats missing as "No" would be wrong. These questions are best analysed **among people who were asked them**.

</details>

In [ ]:
# Your code here

## Part 5. The measurements

### Label each measure

Several concept IDs mean the same measure (two for systolic blood pressure, three or four for LDL). A **lookup table** gives each concept ID its short name, and `left_join()` adds it (joins are the topic of **2.3**).

In [ ]:
measure_lookup <- tribble(
  ~measurement_concept_id, ~measure,
  3038553,  "BMI",
  3004249,  "SBP",      # systolic blood pressure
  4152194,  "SBP",
  3004410,  "HbA1c",
  3028437,  "LDL",
  3028288,  "LDL",
  3009966,  "LDL",
  36031404, "LDL"
)

measurement_labelled <- measurement_raw %>%
  left_join(measure_lookup, by = "measurement_concept_id")

measurement_labelled %>% count(measure)

**🔍 Read the output:** four measures, and no `NA` (every concept ID in the data is in the lookup). If `NA` appeared, a concept ID would be missing from `measure_lookup`.

### Look at the values for each unit

Before trusting or dropping a unit, look at what the **values** look like with it. For each measure and unit: the number of results, the median, and the 5th and 95th percentiles (`quantile()`).

In [ ]:
unit_check <- measurement_labelled %>%
  group_by(measure, unit) %>%
  summarise(n      = n(),
            median = median(value_as_number, na.rm = TRUE),
            p05    = quantile(value_as_number, 0.05, na.rm = TRUE),
            p95    = quantile(value_as_number, 0.95, na.rm = TRUE),
            .groups = "drop") %>%
  mutate(across(c(median, p05, p95), ~ round(.x, 1))) %>%
  arrange(measure, desc(n))

unit_check %>% filter(n > 20) %>% print(n = 60)

**🔍 Read the output:** compare each unit's typical values with the measure's normal range:

| Measure | Usual values in adults |
|---|---|
| BMI | about 18 to 40 kg/m² |
| SBP (systolic blood pressure) | about 90 to 180 mm Hg |
| HbA1c | about 4.5% to 12% |
| LDL | about 50 to 200 mg/dL |

What to look for:

- **"No matching concept"** and `NA` units: if their values look like the measure's usual values, the unit simply wasn't mapped. **Keep them** (with the range check below).
- **Obvious mapping errors** with normal-looking values (in our test run, BMI with the unit *"link for Gunter's chain (US)"*): the values are fine; only the unit label is wrong. **Keep.**
- **A unit that looks wrong, but isn't:** BMI in **percent** sounds like a child's BMI-for-age percentile. In our test run, though, its values looked like ordinary adult BMIs (median 28.6, most between 19 and 39), so it's a **labelling** problem and we **keep** it. Had the values spread evenly from 0 to 100, we'd have dropped them.
- **A different measure under the same name:** HbA1c in **gram per deciliter** (in our test run, a median of 1.1 but values up to 13: a mix, possibly including total hemoglobin) and HbA1c in **milligram per deciliter** (median 132: probably an *estimated average glucose*, not an HbA1c). Some of these would pass a range check, so we **drop them by unit**.

### The cleaning rules

1. **Drop units that mean a different measure:** HbA1c in *gram per deciliter* or *milligram per deciliter*. (If your unit check shows other problems, add them here.)
2. **Keep plausible values only.** Ranges wide enough for real (sick) adults, narrow enough to exclude typing errors, chosen **before** any analysis:

| Measure | Keep from | to |
|---|---|---|
| BMI | 12 | 80 |
| SBP | 60 | 260 |
| HbA1c | 3 | 20 |
| LDL | 10 | 400 |

3. **Remove exact duplicate rows** (same person, measure, date, and value): the same result recorded twice.

Each step is counted, to make a **cleaning log**.

In [ ]:
WRONG_UNITS <- tribble(
  ~measure, ~unit,
  "HbA1c",  "gram per deciliter",
  "HbA1c",  "milligram per deciliter"
)

RANGES <- tribble(
  ~measure, ~low, ~high,
  "BMI",    12,   80,
  "SBP",    60,   260,
  "HbA1c",  3,    20,
  "LDL",    10,   400
)

step0 <- measurement_labelled
step1 <- step0 %>% anti_join(WRONG_UNITS, by = c("measure", "unit"))         # 1. wrong units
step2 <- step1 %>%
  filter(!is.na(value_as_number)) %>%                                         # 2a. no value at all
  left_join(RANGES, by = "measure") %>%
  filter(between(value_as_number, low, high)) %>%                             # 2b. implausible values
  select(-low, -high)
step3 <- step2 %>% distinct(person_id, measure, measurement_date, value_as_number, .keep_all = TRUE)   # 3. duplicates

cleaning_log <- tibble(
  step = c("Start", "1. Drop units that mean another measure", "2. Keep plausible values", "3. Remove exact duplicates"),
  rows = c(nrow(step0), nrow(step1), nrow(step2), nrow(step3))
) %>%
  mutate(removed = lag(rows) - rows)

cleaning_log

**🔍 Read the output:** the **cleaning log**: rows left after each step, and how many each step removed. Report a table like this with your results.

- `anti_join()` keeps rows that **don't** match the wrong-unit list (more in **2.3**).
- Step 2 removes results with no value and values outside the range.
- Step 3 can remove a surprising number: the same result can be recorded more than once in the EHR (e.g. by two systems). In our test run, **235,204** rows (13%) were exact duplicates, far more than the 7,138 implausible values.

Now each measure, after cleaning, by measure:

In [ ]:
measurement_clean <- step3 %>%
  select(person_id, measure, measurement_date, value_as_number, unit)

measurement_clean %>%
  group_by(measure) %>%
  summarise(rows = n(), people = n_distinct(person_id),
            median = round(median(value_as_number), 1),
            min = min(value_as_number), max = max(value_as_number))

save_data(measurement_clean, "measurement_clean.csv")

**🔍 Read the output:** the medians should now look like normal adult values, with min and max inside the ranges. `people` shows how many of our 10,000 have at least one value for each measure: BMI and blood pressure for nearly everyone, HbA1c and LDL for fewer (they're only measured when a doctor orders them).

### ✍️ Your turn

What if we'd used a **narrower** BMI range, 15 to 60? How many more rows would that remove? Would it change the median?

<details>
<summary><b>Show answer</b></summary>

```r
bmi <- measurement_clean %>% filter(measure == "BMI")
c(rows_now = nrow(bmi),
  rows_narrower = sum(between(bmi$value_as_number, 15, 60)),
  median_now = median(bmi$value_as_number),
  median_narrower = median(bmi$value_as_number[between(bmi$value_as_number, 15, 60)]))
```

A small number of rows, and almost no change to the median: the median is **robust** to extreme values. A mean, or a model with BMI as a predictor, can be more sensitive. Whatever range you choose, choose it before the analysis and report it.

</details>

In [ ]:
# Your code here

## Part 6. Missing data

Missing values aren't a cleaning error to fix: they're part of the data. What matters is how you handle them **in each analysis**.

How many of our 10,000 people have each piece of data?

In [ ]:
has <- cohort_clean %>%
  transmute(person_id, t2d,
            BMI   = person_id %in% measurement_clean$person_id[measurement_clean$measure == "BMI"],
            SBP   = person_id %in% measurement_clean$person_id[measurement_clean$measure == "SBP"],
            HbA1c = person_id %in% measurement_clean$person_id[measurement_clean$measure == "HbA1c"],
            LDL   = person_id %in% measurement_clean$person_id[measurement_clean$measure == "LDL"],
            PTSD_answer = person_id %in% survey_final$person_id)

has %>%
  group_by(t2d) %>%
  summarise(across(BMI:PTSD_answer, ~ round(100 * mean(.x), 1)))

# What would "drop every incomplete row" leave?
has %>% summarise(all_four_measures = sum(BMI & SBP & HbA1c & LDL),
                  all_four_and_PTSD = sum(BMI & SBP & HbA1c & LDL & PTSD_answer))

**🔍 Read the output:**

- The first table: the percent of each group with each measure. **Cases** have HbA1c and LDL much more often: their diabetes is monitored. That's a warning sign: missingness is **related to the outcome**, so analysing only people with HbA1c would compare monitored cases with a selected group of controls.
- The second: how many would be left if we required **everything**. Requiring the PTSD answers would leave only a few hundred people, out of 10,000.

**Decision:** we **keep** `NA`s in the files. Each analysis (3.1–3.4) uses the people who have **its** variables, and says how many that is. Never use `drop_na()` on a whole table without checking what it removes.

**💬 Discuss:** a person with no LDL result: does that mean their LDL is normal? (No: it means nobody measured it, or it was measured somewhere that doesn't share data. Missing data can be informative, but it's not a value.)

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *'cohort_sample.csv' does not exist* | 1.1 hasn't been run | Run 1.1 |
| A recoded column is all `NA` | The pattern or list didn't match the real labels | `count()` the original column and compare the exact spelling |
| *`by` must be supplied …* or a join gives too many rows | The join key is wrong or not unique | See **2.3** |
| *cannot allocate vector* | The app ran out of memory | Restart the kernel, and run only this notebook |

## ✅ Key takeaways

- **Look first:** `count()` every label and summarise every value before writing rules
- Non-answers become **`NA`**, never "No"
- A **unit** label alone can't be trusted; check values unit by unit
- Choose **plausible ranges** before the analysis, and keep a **cleaning log**
- Keep missing values; decide how to handle them **per analysis**

**Next:** **2.2 reshape**: from many rows per person to one row per person.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).